# 010-dense-crf-refinement

# Imports

In [ ]:
import itertools
import json
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.datasets import dataset_ref
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.boundary import label_band
from uavloc.segmentation.eval import (MEAN_KEY, VAL_SETS, bilinear_argmax, evaluate, evaluate_predictions, load_baseline_model,
                                      load_val_sets, lraspp_outputs, results_table, run_model)
from uavloc.segmentation.latency import refiner_latency, standard_tile
from uavloc.segmentation.refine.dense_crf import DenseCrfConfig, dense_crf_refiner
from uavloc.track import experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "010-dense-crf-refinement"
NOTE = ""
LOG_RUN = True

MODE = "grid"                    # "baseline" | "grid" (tune on TUNE_SET, score the pick) | "fixed" (FIXED_CONFIG)
TUNE_SET = "birky-strip"
HELD_OUT_SET = "bushcha"
GRID_ITERATIONS = [5, 10]
GRID_BILATERAL_SXY_PX = [20, 40, 80]
GRID_BILATERAL_SRGB = [5, 13]
GRID_BILATERAL_COMPAT = [3, 10]
GAUSSIAN_SXY_PX = 3.0
GAUSSIAN_COMPAT = 3.0
MAX_MIOU_DROP = 0.01
TIE_BIOU = 0.001
FIXED_CONFIG = {"iterations": 5, "bilateral_sxy_px": 80.0, "bilateral_srgb": 13.0, "bilateral_compat": 10.0}

DATASET = "ua-village-annotated"
DEVICE = "cpu"
SEED = 42
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_THREADS = 4
LATENCY_WARMUP = 2
LATENCY_RUNS = 10
ZOOM_CROPS = [["birky-strip-google", 0, 120, 1400, 376, 1912], ["bushcha", 0, 1024, 1024, 1280, 1536]]

# Data

In [ ]:
set_seed(SEED)
FIGURES = experiment_dir(EXP) / "figures"
ARTIFACTS = experiment_dir(EXP) / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
val_sets = load_val_sets(DATASET)
{name: [a.rgb.shape for a in items] for name, items in val_sets.items()}

# Model

In [ ]:
model = load_baseline_model(DEVICE)

In [ ]:
cases = run_model(model, val_sets, DEVICE)

# Run

In [ ]:
def grid_configs() -> list[DenseCrfConfig]:
    """The registered 24-config grid (proposal.md)."""
    return [DenseCrfConfig(iterations=it, bilateral_sxy_px=float(sxy), bilateral_srgb=float(srgb),
                           bilateral_compat=float(compat), gaussian_sxy_px=GAUSSIAN_SXY_PX,
                           gaussian_compat=GAUSSIAN_COMPAT)
            for it, sxy, srgb, compat in itertools.product(GRID_ITERATIONS, GRID_BILATERAL_SXY_PX,
                                                          GRID_BILATERAL_SRGB, GRID_BILATERAL_COMPAT)]

In [ ]:
def tune_row(config: DenseCrfConfig) -> dict:
    """The config's scalar scores on TUNE_SET only."""
    scores = evaluate(dense_crf_refiner(config), {TUNE_SET: cases[TUNE_SET]})[TUNE_SET]
    return {**asdict(config), **{k: v for k, v in scores.items() if not isinstance(v, dict)}}

In [ ]:
def select(grid: pd.DataFrame, baseline_miou: float) -> pd.Series:
    """Best TUNE_SET Boundary IoU within the mIoU guard; ties go to fewer iterations, then smaller sxy."""
    ok = grid[grid["miou"] >= baseline_miou - MAX_MIOU_DROP]
    if ok.empty:
        raise ValueError("no grid config passes the mIoU guard on the tuning set")
    near_best = ok[ok["biou"] >= ok["biou"].max() - TIE_BIOU]
    return near_best.sort_values(["iterations", "bilateral_sxy_px", "biou"], ascending=[True, True, False]).iloc[0]

In [ ]:
baseline_tune = evaluate(bilinear_argmax, {TUNE_SET: cases[TUNE_SET]})[TUNE_SET]
grid = pd.DataFrame([tune_row(c) for c in grid_configs()]) if MODE == "grid" else pd.DataFrame()
grid.sort_values("biou", ascending=False).round(4) if MODE == "grid" else baseline_tune["biou"]

In [ ]:
def config_from(values: dict) -> DenseCrfConfig:
    """A CRF config from a grid row or FIXED_CONFIG; the Gaussian kernel is fixed by Params."""
    return DenseCrfConfig(iterations=int(values["iterations"]), bilateral_sxy_px=float(values["bilateral_sxy_px"]),
                          bilateral_srgb=float(values["bilateral_srgb"]),
                          bilateral_compat=float(values["bilateral_compat"]),
                          gaussian_sxy_px=GAUSSIAN_SXY_PX, gaussian_compat=GAUSSIAN_COMPAT)

In [ ]:
if MODE == "baseline":
    config, refine_fn = None, bilinear_argmax
else:
    config = config_from(select(grid, baseline_tune["miou"]) if MODE == "grid" else FIXED_CONFIG)
    refine_fn = dense_crf_refiner(config)
config

In [ ]:
preds = {name: [refine_fn(c.output, c.image.rgb) for c in items] for name, items in cases.items()}
results = evaluate_predictions(preds, cases)
results_table(results).round(4)

In [ ]:
per_class = pd.concat({m: pd.DataFrame({n: results[n][f"{m}_per_class"] for n in VAL_SETS}).T
                       for m in ["iou", "biou", "bf1_1px", "bf1_2px"]}, axis=1)
per_class.round(3)

In [ ]:
latency_tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
latency_output = lraspp_outputs(model, latency_tile, DEVICE)
latency = refiner_latency(refine_fn, latency_output, latency_tile, warmup=LATENCY_WARMUP, runs=LATENCY_RUNS,
                          threads=LATENCY_THREADS)
latency

In [ ]:
metrics = {**{f"{k}_mean": v for k, v in results[MEAN_KEY].items()},
           **{f"{k}_{name}": results[name][k] for name in VAL_SETS for k in ("miou", "biou", "bf1_1px", "bf1_2px")},
           "biou_per_class": {name: results[name]["biou_per_class"] for name in VAL_SETS},
           "iou_per_class": {name: results[name]["iou_per_class"] for name in VAL_SETS},
           "refine_cpu_ms_512": latency.median_ms, "refine_cpu_ms_512_p10": latency.p10_ms,
           "refine_cpu_ms_512_p90": latency.p90_ms}
{k: round(v, 4) for k, v in metrics.items() if not isinstance(v, dict)}

# Plots

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != IGNORE_LABEL
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
def band_overlay(pred: np.ndarray, label: np.ndarray) -> np.ndarray:
    """Predicted classes, with the label's 1-px class contours drawn in white."""
    out = colour(pred)
    out[np.any([label_band(label, c, 1) for c in range(len(CLASSES))], axis=0)] = 255
    return out

In [ ]:
def zoom_panels(set_name: str, index: int, top: int, left: int, bottom: int, right: int) -> list[tuple]:
    case = cases[set_name][index]
    zoom = np.s_[top:bottom, left:right]
    lab = case.image.label[zoom]
    return [(case.image.rgb[zoom], f"{set_name} RGB"), (colour(lab), "label"),
            (band_overlay(bilinear_argmax(case.output, case.image.rgb)[zoom], lab), "baseline"),
            (band_overlay(preds[set_name][index][zoom], lab), MODE)]

In [ ]:
rows = [zoom_panels(*crop) for crop in ZOOM_CROPS]
fig_zoom, axes = plt.subplots(len(rows), len(rows[0]), figsize=(5 * len(rows[0]), 2.8 * len(rows)), squeeze=False)
for ax_row, panels in zip(axes, rows):
    for ax, (img, title) in zip(ax_row, panels):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title); ax.axis("off")
fig_zoom.tight_layout()

In [ ]:
def grid_heatmap(grid: pd.DataFrame, baseline_biou: float) -> plt.Figure:
    """TUNE_SET Boundary IoU minus baseline, per (iterations, compat) panel over sxy x srgb."""
    panels = list(itertools.product(GRID_ITERATIONS, GRID_BILATERAL_COMPAT))
    fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3), sharey=True)
    for ax, (it, compat) in zip(axes, panels):
        sub = grid[(grid["iterations"] == it) & (grid["bilateral_compat"] == compat)]
        table = sub.pivot(index="bilateral_sxy_px", columns="bilateral_srgb", values="biou") - baseline_biou
        ax.imshow(table.to_numpy(), cmap="RdBu", vmin=-0.03, vmax=0.03)
        for (i, j), v in np.ndenumerate(table.to_numpy()):
            ax.text(j, i, f"{v:+.3f}", ha="center", va="center", fontsize=8)
        ax.set_xticks(range(len(table.columns)), table.columns); ax.set_yticks(range(len(table.index)), table.index)
        ax.set_xlabel("srgb"); ax.set_title(f"{it} it, compat {compat:g}")
    axes[0].set_ylabel("bilateral sxy, px")
    fig.suptitle(f"{TUNE_SET} Boundary IoU - baseline"); fig.tight_layout()
    return fig

In [ ]:
figures = [fig_zoom] + ([grid_heatmap(grid, baseline_tune["biou"])] if MODE == "grid" else [])

# Log

In [ ]:
RUN_CONFIG = {"mode": MODE, "refiner": "bilinear_argmax" if config is None else "dense_crf",
              "crf": None if config is None else asdict(config), "dataset": dataset_ref(DATASET),
              "tune_set": TUNE_SET, "grid": {"iterations": GRID_ITERATIONS, "bilateral_sxy_px": GRID_BILATERAL_SXY_PX,
                                             "bilateral_srgb": GRID_BILATERAL_SRGB,
                                             "bilateral_compat": GRID_BILATERAL_COMPAT} if MODE == "grid" else None,
              "latency": {"set": LATENCY_SET, "top_left_px": LATENCY_TOP_LEFT_PX, "threads": LATENCY_THREADS,
                          "runs": LATENCY_RUNS}}

In [ ]:
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=figures, seed=SEED,
              device=DEVICE) if LOG_RUN else {"run": "dry"}
if MODE == "grid":
    grid.to_csv(ARTIFACTS / f"grid-{run['run']}.csv", index=False)
(ARTIFACTS / f"results-{run['run']}.json").write_text(json.dumps(results, indent=1, default=str))
run["run"]